In [1]:
!pip install "datasets<3.0.0" soundfile

!pip install transformers accelerate torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 3.6 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.


In [2]:
!pip install resemblyzer  # lightweight speaker embedding library


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 3.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 3.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 35.3 MB/s eta 0:00:00
  Created wheel for webrtcvad: filename=webrtcvad-2.0.10-cp312-cp312-linux_x86_64.whl size=73516 sha256=c035f7c016eef5774fa43ab0d1c3a4beb4f0eb1e98c9e6f20c4beeeb42bb05cf
  Stored in directory: /root/.cache/pip/wheels/1e/d3/95/680fa3b16848f1a58d2edaed34c496224c89a9bc63e17b3614
  Created wheel for typing: filename=typing-3.7.4.3-py3-none-any.whl size=26304 sha256=65cb2265ae51aeb3fca3998d3f1a0480be00aa807ed425a9fe5d0e6b7378e0e4
  Stored in directory: /root/.cache/pip/wheels/12/98/52/2bffe242a9a487f00886e43b8ed8dac46456702e11a0d6abef
Successfully built webrtcvad typing


In [3]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
from datasets import load_dataset

# Target Urdu directory
ds_urduspeech_stream = load_dataset(
    "humairawan/UrduSpeech",
    data_dir="Urdu",
    split="train",
    token='',
    streaming=True
)

# Take first NUM_SAMPLES with style = "CONV"
urduspeech_samples = []
for sample in ds_urduspeech_stream:
    if sample.get("style") == "CONV" and sample.get("gender") == "Male": # filter conversational
        urduspeech_samples.append(sample)
        print(f"Collected samples: {len(urduspeech_samples)}")
    if len(urduspeech_samples) >= 60:
        break # first sample

Resolving data files:   0%|          | 0/56 [00:00<?, ?it/s]

Collected samples: 1
Collected samples: 2
Collected samples: 3
Collected samples: 4
Collected samples: 5
Collected samples: 6
Collected samples: 7
Collected samples: 8
Collected samples: 9
Collected samples: 10
Collected samples: 11
Collected samples: 12
Collected samples: 13
Collected samples: 14
Collected samples: 15
Collected samples: 16
Collected samples: 17
Collected samples: 18
Collected samples: 19
Collected samples: 20
Collected samples: 21
Collected samples: 22
Collected samples: 23
Collected samples: 24
Collected samples: 25
Collected samples: 26
Collected samples: 27
Collected samples: 28
Collected samples: 29
Collected samples: 30
Collected samples: 31
Collected samples: 32
Collected samples: 33
Collected samples: 34
Collected samples: 35
Collected samples: 36
Collected samples: 37
Collected samples: 38
Collected samples: 39
Collected samples: 40
Collected samples: 41
Collected samples: 42
Collected samples: 43
Collected samples: 44
Collected samples: 45
Collected samples: 

In [5]:
from datasets import load_dataset
import google.generativeai as genai
import io, wave
import numpy as np
import soundfile as sf
from pathlib import Path
from resemblyzer import VoiceEncoder, preprocess_wav
import time

# =========================
# Setup
# =========================
genai.configure(api_key="")
model = genai.GenerativeModel("gemini-2.5-flash-preview-tts")
urdu_conv = urduspeech_samples
encoder = VoiceEncoder()

scores = []
results = []

for i in range(0,10):

    text = urdu_conv[i]['text']
    ref_array = urdu_conv[i]['audio']['array'].astype(np.float32)
    ref_sr = urdu_conv[i]['audio']['sampling_rate']

    ref_path = f"ref_{i}.wav"
    gen_path = f"gen_{i}.wav"

    # Save reference
    sf.write(ref_path, ref_array, samplerate=ref_sr)

    # =========================
    # Gemini TTS
    # =========================
    speech_config = {
        "voice_config": {
            "prebuilt_voice_config": {
                "voice_name": "charon"
            }
        }
    }

    response = model.generate_content(
        f"Please read this text: {text}",
        generation_config={
            "response_modalities": ["AUDIO"],
            "speech_config": speech_config
        }
    )

    try:
        audio_bytes = response.candidates[0].content.parts[0].inline_data.data

        byte_io = io.BytesIO()
        with wave.open(byte_io, 'wb') as wav_file:
            wav_file.setnchannels(1)
            wav_file.setsampwidth(2)
            wav_file.setframerate(24000)
            wav_file.writeframes(audio_bytes)

        byte_io.seek(0)

        with open(gen_path, "wb") as f:
            f.write(byte_io.read())

    except Exception as e:
        print(f"Error on sample {i}: {e}")
        continue
    print(f"Sample {i}: {text[:40]}...")
    # =========================
    # Delay after every 3 iterations
    # =========================
    if (i + 1) % 3 == 0:
        print("Waiting for 1.2 minutes...")
        time.sleep(72)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Loaded the voice encoder model on cpu in 0.18 seconds.
Sample 0: گوا کی یہ تہہ دار میٹھی چیز ہندوستان کی ...
Sample 1: آئیے دیکھتے ہیں کہ حالات ٹھیک رہے تو میں...
Sample 2: میں واقعی اس کی تعریف کرتا ہوں۔...
Waiting for 1.2 minutes...
Sample 3: اس کے اوقات کیا ہیں؟...
Error on sample 4: list index out of range
Sample 5: ذاتی تفصیلات جیسے رابطے اور رہائش کی تفص...
Waiting for 1.2 minutes...
Sample 6: آپ نے حقیقت میں ہمارا ایک بڑا مسئلہ حل ک...
Sample 7: فن آرٹ آن لائن آرٹ کلاس کو فن پارے اور د...
Sample 8: ہے نا! وہ تو زبردست تھا۔...
Waiting for 1.2 minutes...
Sample 9: یہ پی ڈی ایف فارمیٹ میں جاری کیا جائے گا...


In [6]:
from datasets import load_dataset
import google.generativeai as genai
import io, wave
import numpy as np
import soundfile as sf
from pathlib import Path
from resemblyzer import VoiceEncoder, preprocess_wav
import time

# =========================
# Setup
# =========================
genai.configure(api_key="")
model = genai.GenerativeModel("gemini-2.5-flash-preview-tts")
urdu_conv = urduspeech_samples
encoder = VoiceEncoder()

scores = []
results = []

for i in range(10,20):

    text = urdu_conv[i]['text']
    ref_array = urdu_conv[i]['audio']['array'].astype(np.float32)
    ref_sr = urdu_conv[i]['audio']['sampling_rate']

    ref_path = f"ref_{i}.wav"
    gen_path = f"gen_{i}.wav"

    # Save reference
    sf.write(ref_path, ref_array, samplerate=ref_sr)

    # =========================
    # Gemini TTS
    # =========================
    speech_config = {
        "voice_config": {
            "prebuilt_voice_config": {
                "voice_name": "charon"
            }
        }
    }

    response = model.generate_content(
        f"Please read this text: {text}",
        generation_config={
            "response_modalities": ["AUDIO"],
            "speech_config": speech_config
        }
    )

    try:
        audio_bytes = response.candidates[0].content.parts[0].inline_data.data

        byte_io = io.BytesIO()
        with wave.open(byte_io, 'wb') as wav_file:
            wav_file.setnchannels(1)
            wav_file.setsampwidth(2)
            wav_file.setframerate(24000)
            wav_file.writeframes(audio_bytes)

        byte_io.seek(0)

        with open(gen_path, "wb") as f:
            f.write(byte_io.read())

    except Exception as e:
        print(f"Error on sample {i}: {e}")
        continue
    print(f"Sample {i}: {text[:40]}...")
    # =========================
    # Delay after every 3 iterations
    # =========================
    if (i + 1) % 3 == 0:
        print("Waiting for 1.2 minutes...")
        time.sleep(72)

Loaded the voice encoder model on cpu in 0.02 seconds.
Sample 10: دہرادون میں ہمارا سنٹر رائے پور روڈ، دہر...
Sample 11: جی ہاں، وہ بہر حال مہیا کیا جائے گا۔...
Waiting for 1.2 minutes...
Sample 12: میں آپ کی پریشانی کو سمجھتا ہوں۔...
Sample 13: یہ مندروں کا شہر ہے تو شاید آپ مجھے وہاں...
Sample 14: کوئی شخص اپنے تفریحی مشغلوں سے دوسرے کون...
Waiting for 1.2 minutes...
Sample 15: سنجیدگی سے، یہ ضروری ہے کہ ہم دن میں کچھ...
Sample 16: لہذا ہم جو بھی پڑھتے ہیں میں ان کی تصاوی...
Sample 17: یہاں تک کہ میرے جسمانی عمل اور دماغی صحت...
Waiting for 1.2 minutes...
Sample 18: میوزک اور ڈانس کا کیا؟...
Sample 19: ٹھیک ہے۔ یہ کافی مہنگا ہے۔...


In [7]:
from datasets import load_dataset
import google.generativeai as genai
import io, wave
import numpy as np
import soundfile as sf
from pathlib import Path
from resemblyzer import VoiceEncoder, preprocess_wav
import time

# =========================
# Setup
# =========================
genai.configure(api_key="")
model = genai.GenerativeModel("gemini-2.5-flash-preview-tts")
urdu_conv = urduspeech_samples
encoder = VoiceEncoder()

scores = []
results = []

for i in range(20,30):

    text = urdu_conv[i]['text']
    ref_array = urdu_conv[i]['audio']['array'].astype(np.float32)
    ref_sr = urdu_conv[i]['audio']['sampling_rate']

    ref_path = f"ref_{i}.wav"
    gen_path = f"gen_{i}.wav"

    # Save reference
    sf.write(ref_path, ref_array, samplerate=ref_sr)

    # =========================
    # Gemini TTS
    # =========================
    speech_config = {
        "voice_config": {
            "prebuilt_voice_config": {
                "voice_name": "charon"
            }
        }
    }

    response = model.generate_content(
        f"Please read this text: {text}",
        generation_config={
            "response_modalities": ["AUDIO"],
            "speech_config": speech_config
        }
    )

    try:
        audio_bytes = response.candidates[0].content.parts[0].inline_data.data

        byte_io = io.BytesIO()
        with wave.open(byte_io, 'wb') as wav_file:
            wav_file.setnchannels(1)
            wav_file.setsampwidth(2)
            wav_file.setframerate(24000)
            wav_file.writeframes(audio_bytes)

        byte_io.seek(0)

        with open(gen_path, "wb") as f:
            f.write(byte_io.read())

    except Exception as e:
        print(f"Error on sample {i}: {e}")
        continue
    print(f"Sample {i}: {text[:40]}...")
    # =========================
    # Delay after every 3 iterations
    # =========================
    if (i + 1) % 3 == 0:
        print("Waiting for 1.2 minutes...")
        time.sleep(72)

Loaded the voice encoder model on cpu in 0.09 seconds.
Sample 20: متبادل کے طور پر، آپ ڈیبٹ یا کریڈٹ کارڈ ...
Waiting for 1.2 minutes...
Sample 21: سر، تو آپ چاہتے ہیں کہ میں اعلان کروں یا...
Sample 22: اس کی کون سی مختلف اقسام ہیں؟...
Sample 23: علاوہ ازیں، ہندوستانی جغرافیہ ہمارے نصاب...
Waiting for 1.2 minutes...
Sample 24: براہ مہربانی بتائیے۔...
Sample 25: کیا آپ کے لیے یہ ٹھیک رہے گا؟...
Sample 26: براہ کرم مجھے بتائیے کہ آپ کس وقت دستیاب...
Waiting for 1.2 minutes...
Sample 27: جی ہاں، ہم کس طرح آپ کی مدد کر سکتے ہیں؟...
Sample 28: اگر آپ کو کسی مدد کی ضرورت ہو تو مجھ سے ...
Error on sample 29: list index out of range


In [9]:
from datasets import load_dataset
import google.generativeai as genai
import io, wave
import numpy as np
import soundfile as sf
from pathlib import Path
from resemblyzer import VoiceEncoder, preprocess_wav
import time

# =========================
# Setup
# =========================
genai.configure(api_key="")
model = genai.GenerativeModel("gemini-2.5-flash-preview-tts")
urdu_conv = urduspeech_samples
encoder = VoiceEncoder()

scores = []
results = []

for i in range(35,40):

    text = urdu_conv[i]['text']
    ref_array = urdu_conv[i]['audio']['array'].astype(np.float32)
    ref_sr = urdu_conv[i]['audio']['sampling_rate']

    ref_path = f"ref_{i}.wav"
    gen_path = f"gen_{i}.wav"

    # Save reference
    sf.write(ref_path, ref_array, samplerate=ref_sr)

    # =========================
    # Gemini TTS
    # =========================
    speech_config = {
        "voice_config": {
            "prebuilt_voice_config": {
                "voice_name": "charon"
            }
        }
    }

    response = model.generate_content(
        f"Please read this text: {text}",
        generation_config={
            "response_modalities": ["AUDIO"],
            "speech_config": speech_config
        }
    )

    try:
        audio_bytes = response.candidates[0].content.parts[0].inline_data.data

        byte_io = io.BytesIO()
        with wave.open(byte_io, 'wb') as wav_file:
            wav_file.setnchannels(1)
            wav_file.setsampwidth(2)
            wav_file.setframerate(24000)
            wav_file.writeframes(audio_bytes)

        byte_io.seek(0)

        with open(gen_path, "wb") as f:
            f.write(byte_io.read())

    except Exception as e:
        print(f"Error on sample {i}: {e}")
        continue
    print(f"Sample {i}: {text[:40]}...")
    # =========================
    # Delay after every 3 iterations
    # =========================
    if (i + 1) % 3 == 0:
        print("Waiting for 1.2 minutes...")
        time.sleep(72)

Loaded the voice encoder model on cpu in 0.04 seconds.
Sample 35: میں خود بھی سوچتا رہتا ہوں، کہ کیوں اتنا...
Waiting for 1.2 minutes...
Error on sample 36: list index out of range
Sample 37: ویسے کیا ہم سب صرف جنک نہیں ہیں؟...
Sample 38: واقعی نہیں، شاید ہم مل کر گھوم سکتے ہیں۔...
Waiting for 1.2 minutes...
Sample 39: کیا آپ میرے کچھ سوالات کا جواب دے سکتے ہ...


In [10]:
from datasets import load_dataset
import google.generativeai as genai
import io, wave
import numpy as np
import soundfile as sf
from pathlib import Path
from resemblyzer import VoiceEncoder, preprocess_wav
import time

# =========================
# Setup
# =========================
genai.configure(api_key="")
model = genai.GenerativeModel("gemini-2.5-flash-preview-tts")
urdu_conv = urduspeech_samples
encoder = VoiceEncoder()

scores = []
results = []

for i in range(40,50):

    text = urdu_conv[i]['text']
    ref_array = urdu_conv[i]['audio']['array'].astype(np.float32)
    ref_sr = urdu_conv[i]['audio']['sampling_rate']

    ref_path = f"ref_{i}.wav"
    gen_path = f"gen_{i}.wav"

    # Save reference
    sf.write(ref_path, ref_array, samplerate=ref_sr)

    # =========================
    # Gemini TTS
    # =========================
    speech_config = {
        "voice_config": {
            "prebuilt_voice_config": {
                "voice_name": "charon"
            }
        }
    }

    response = model.generate_content(
        f"Please read this text: {text}",
        generation_config={
            "response_modalities": ["AUDIO"],
            "speech_config": speech_config
        }
    )

    try:
        audio_bytes = response.candidates[0].content.parts[0].inline_data.data

        byte_io = io.BytesIO()
        with wave.open(byte_io, 'wb') as wav_file:
            wav_file.setnchannels(1)
            wav_file.setsampwidth(2)
            wav_file.setframerate(24000)
            wav_file.writeframes(audio_bytes)

        byte_io.seek(0)

        with open(gen_path, "wb") as f:
            f.write(byte_io.read())

    except Exception as e:
        print(f"Error on sample {i}: {e}")
        continue
    print(f"Sample {i}: {text[:40]}...")
    # =========================
    # Delay after every 3 iterations
    # =========================
    if (i + 1) % 3 == 0:
        print("Waiting for 1.2 minutes...")
        time.sleep(72)

Loaded the voice encoder model on cpu in 0.02 seconds.
Sample 40: مجھے بتاؤ، وویک، آپ کا مسئلہ کیا ہے؟...
Sample 41: میں ویشنو دیوی مندر کی زیارت کے بارے میں...
Waiting for 1.2 minutes...
Sample 42: یہ واقعی ایک تجربہ ہوتا ہے۔...
Sample 43: یہاں تک کہ یہ بھی زیادہ نہیں ہے۔...
Sample 44: براہ کرم مجھے بتائیے آپ کو کیا معلوم کرن...
Waiting for 1.2 minutes...
Sample 45: ایس بی آئی انٹرنیٹ بینکنگ فراہم کرتا ہے۔...
Sample 46: جی ایچ آئی نگر، فلاں فلاں اسٹریٹ، فلاں ف...
Sample 47: کیا یہ بات کرنے کا صحیح وقت ہے؟...
Waiting for 1.2 minutes...
Error on sample 48: list index out of range
Sample 49: ہاں، سر، میں تقریبا ہر چیز پر سے اپنی تو...


In [12]:
from datasets import load_dataset
import google.generativeai as genai
import io, wave
import numpy as np
import soundfile as sf
from pathlib import Path
from resemblyzer import VoiceEncoder, preprocess_wav
import time

# =========================
# Setup
# =========================
genai.configure(api_key="")
model = genai.GenerativeModel("gemini-2.5-flash-preview-tts")
urdu_conv = urduspeech_samples
encoder = VoiceEncoder()

scores = []
results = []

for i in range(55,60):

    text = urdu_conv[i]['text']
    ref_array = urdu_conv[i]['audio']['array'].astype(np.float32)
    ref_sr = urdu_conv[i]['audio']['sampling_rate']

    ref_path = f"ref_{i}.wav"
    gen_path = f"gen_{i}.wav"

    # Save reference
    sf.write(ref_path, ref_array, samplerate=ref_sr)

    # =========================
    # Gemini TTS
    # =========================
    speech_config = {
        "voice_config": {
            "prebuilt_voice_config": {
                "voice_name": "charon"
            }
        }
    }

    response = model.generate_content(
        f"Please read this text: {text}",
        generation_config={
            "response_modalities": ["AUDIO"],
            "speech_config": speech_config
        }
    )

    try:
        audio_bytes = response.candidates[0].content.parts[0].inline_data.data

        byte_io = io.BytesIO()
        with wave.open(byte_io, 'wb') as wav_file:
            wav_file.setnchannels(1)
            wav_file.setsampwidth(2)
            wav_file.setframerate(24000)
            wav_file.writeframes(audio_bytes)

        byte_io.seek(0)

        with open(gen_path, "wb") as f:
            f.write(byte_io.read())

    except Exception as e:
        print(f"Error on sample {i}: {e}")
        if (i + 1) % 3 == 0:
          print("Waiting for 1.2 minutes...")
          time.sleep(72)
        continue
    print(f"Sample {i}: {text[:40]}...")
    # =========================
    # Delay after every 3 iterations
    # =========================
    if (i + 1) % 3 == 0:
        print("Waiting for 1.2 minutes...")
        time.sleep(72)

Loaded the voice encoder model on cpu in 0.02 seconds.
Sample 55: ہم آپ کے لیے کیا کر سکتے ہیں؟...
Sample 56: بالکل! اب آپ کو اتنا کرنا ہے کہ آپ اسے ل...
Waiting for 1.2 minutes...
Sample 57: وہاں پر ساؤتھ انڈیا کے مسالا دوسا سے لے ...
Sample 58: بالکل، مجھے جلدی نہیں ہے۔...
Error on sample 59: list index out of range
Waiting for 1.2 minutes...


In [22]:
from datasets import load_dataset
import google.generativeai as genai
import io, wave
import numpy as np
import soundfile as sf
from pathlib import Path
from resemblyzer import VoiceEncoder, preprocess_wav
import time

# =========================
# Setup
# =========================
genai.configure(api_key="")
model = genai.GenerativeModel("gemini-2.5-flash-preview-tts")
urdu_conv = urduspeech_samples
encoder = VoiceEncoder()

scores = []
results = []

for i in [32]: #32

    text = urdu_conv[i]['text']
    ref_array = urdu_conv[i]['audio']['array'].astype(np.float32)
    ref_sr = urdu_conv[i]['audio']['sampling_rate']

    ref_path = f"ref_{i}.wav"
    gen_path = f"gen_{i}.wav"

    # Save reference
    sf.write(ref_path, ref_array, samplerate=ref_sr)

    # =========================
    # Gemini TTS
    # =========================
    speech_config = {
        "voice_config": {
            "prebuilt_voice_config": {
                "voice_name": "charon"
            }
        }
    }

    response = model.generate_content(
      f"Read this aloud in a clear and natural voice:\n{text}",
      generation_config={
          "response_modalities": ["AUDIO"],
          "speech_config": speech_config
      }
  )

    try:
        audio_bytes = response.candidates[0].content.parts[0].inline_data.data

        byte_io = io.BytesIO()
        with wave.open(byte_io, 'wb') as wav_file:
            wav_file.setnchannels(1)
            wav_file.setsampwidth(2)
            wav_file.setframerate(24000)
            wav_file.writeframes(audio_bytes)

        byte_io.seek(0)

        with open(gen_path, "wb") as f:
            f.write(byte_io.read())

    except Exception as e:
        print(f"Error on sample {i}: {e}")
        continue
    print(f"Sample {i}: {text[:40]}...")


Loaded the voice encoder model on cpu in 0.01 seconds.


TooManyRequests: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash-preview-tts:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 10, model: gemini-2.5-flash-tts
Please retry in 16.879905423s.

In [25]:
print(urdu_conv[32]['text'])

ارے کرش!


In [23]:
import wave
import numpy as np

filename = "gen_32.wav"

sample_rate = 22050
duration = 1  # seconds

silence = np.zeros(sample_rate * duration, dtype=np.int16)

with wave.open(filename, "w") as wav_file:
    wav_file.setnchannels(1)
    wav_file.setsampwidth(2)
    wav_file.setframerate(sample_rate)
    wav_file.writeframes(silence.tobytes())

In [24]:
# ============================================
# PHASE 2: Package audio files for MUSHRA
# ============================================
import os
import zipfile

# Create folder structure webMUSHRA expects
os.makedirs("mushra_audio/reference", exist_ok=True)
os.makedirs("mushra_audio/generated_gemini_tts", exist_ok=True)


for i in range(60):
    # Copy reference
    ref_src = f"ref_{i}.wav"
    gen_src = f"gen_{i}.wav"

    # Read files
    ref_data, ref_sr = sf.read(ref_src)
    gen_data, gen_sr = sf.read(gen_src)

    # Save into structured folders
    sf.write(f"mushra_audio/reference/sample_{i}.wav", ref_data, ref_sr)
    sf.write(f"mushra_audio/generated_gemini_tts/sample_{i}.wav", gen_data, gen_sr)



# Zip everything for download
with zipfile.ZipFile("60_mushra_conv_gemini_audio.zip", "w") as zf:
    for root, dirs, files in os.walk("mushra_audio"):
        for file in files:
            zf.write(os.path.join(root, file))

print("60_mushra_audio.zip ready for download!")

# Download in Colab
from google.colab import files
files.download("60_mushra_conv_gemini_audio.zip")

60_mushra_audio.zip ready for download!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>